In [25]:
import tidy3d as td
import numpy as np
import pandas as pd
print(td.__version__)

import matplotlib.pyplot as plt
# %matplotlib widget
from matplotlib.colors import LogNorm

# import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

import gc
import os

import tidy3d.web as web
from getpass import getpass
api_key = getpass("Enter your API key: ")
# web.configure("_blank_")
web.configure(api_key)

2.10.2


Enter your API key:  ········


Configuration saved successfully.


In [2]:
web.test()

17:58:58 Malay Peninsula Standard Time Authentication configured successfully!

# Definitions

In [3]:
h_planck = 6.62607015e-34
e_charge = 1.602176634e-19
def hz_to_ev(f): return (h_planck * f) / e_charge
def ev_to_hz(E): return (E * e_charge) / h_planck

In [4]:
def extract_monitor_data(filepath, monitor_name):
    
    # Load full simulation (unavoidable)
    sim_data = td.SimulationData.from_file(filepath)
    
    # Extract ONLY the monitor we need
    monitor_data = sim_data[monitor_name]
    
    # Delete the full simulation data immediately
    del sim_data
    gc.collect()
    
    return monitor_data

In [5]:
def get_plane_config(plane):
    """Return slicing info, labels, and field pairs for a given plane."""
    configs = {
        'Exy': {'slicer': {'z': 0},
            'U_field': 'Ex', 'V_field': 'Ey',
            'xlabel': 'x (nm)', 'ylabel': 'y (nm)',
            'coord_keys': ('x', 'y')},
        
        'Exz': {'slicer': {'y': 0},
            'U_field': 'Ex', 'V_field': 'Ez',
            'xlabel': 'x (nm)', 'ylabel': 'z (nm)',
            'coord_keys': ('x', 'z')},
        
        'Eyz': {'slicer': {'x': 0},
            'U_field': 'Ey', 'V_field': 'Ez',
            'xlabel': 'y (nm)', 'ylabel': 'z (nm)',
            'coord_keys': ('y', 'z')},
    }
    if plane not in configs:
        raise ValueError(f"plane must be one of {list(configs.keys())}")
    
    return configs[plane]


# ========================================


In [6]:
def get_field_component(monitor_data, monitor_data0, comp, slicer, peak, normalize):
    """Extract field component from monitor data"""
    data  = getattr(monitor_data , comp).isel(**slicer).interp(f=peak)
    data0 = getattr(monitor_data0, comp).isel(**slicer).interp(f=peak)
    
    if normalize:
        return (data - data0) / data0
    else:
        return data - data0


def get_coords(monitor_data, plane):
    """Extract and format coordinates for plotting"""
    coord_key1, coord_key2 = get_plane_config(plane)['coord_keys']
    
    # Extract coordinates and convert to nm
    coord1 = monitor_data.Ex.coords[coord_key1].values * 1e3
    coord2 = monitor_data.Ex.coords[coord_key2].values * 1e3
    
    # Ensure increasing order
    if coord1[0] > coord1[-1]:  # if first is larger than last
        coord1 = coord1[::-1]   # reverse all
    if coord2[0] > coord2[-1]:
        coord2 = coord2[::-1]
    
    return coord1, coord2


# ========================================


In [13]:
def prepare_Efield_data(Ex, Ey, Ez, coord1, coord2, plane):
    """Compute |E| and meshgrid from field components"""
    # Convert to real NumPy arrays
    Ex = np.real(np.array(Ex))
    Ey = np.real(np.array(Ey))
    Ez = np.real(np.array(Ez))
    
    # Expected shape
    expected_shape = (len(coord2), len(coord1))
    
    # Transpose if needed
    if Ex.shape != expected_shape:
        print(f"(Ex.shape={Ex.shape}, expected={expected_shape})")
        Ex, Ey, Ez = Ex.T, Ey.T, Ez.T
    
    # For xy plane, apply additional transpose
    # if plane == 'Exy':
    #     Ex, Ey, Ez = Ex.T, Ey.T, Ez.T
    #     print("[INFO] Applied xy-plane transpose for correct orientation")
    
    # Compute magnitude
    E = np.sqrt(np.abs(Ex)**2 + np.abs(Ey)**2 + np.abs(Ez)**2)
    
    # Build coordinate mesh
    horizontal_axis, vertical_axis = np.meshgrid(coord1, coord2)

    # Orientation diagnostics
    print(f"[DEBUG] {plane}-plane orientation check:")
    print(f"  horizontal_axis shape={horizontal_axis.shape}, "
          f"vertical_axis shape={vertical_axis.shape}")
    print(f"  H-field array shape={E.shape}")
    
    return E, horizontal_axis, vertical_axis, Ex, Ey, Ez


# ========================================


In [8]:
def plot_Efield(E, horizontal_axis, vertical_axis, U, V, 
               xlabel, ylabel, plane, peak, name, monitor,
               fixed_scale, density, arrow, save_path=None):

    """Render field magnitude and streamlines"""
    # plt.figure(figsize=(8, 6))
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(f"{name} | {monitor} | {hz_to_ev(peak):.3f} eV")
    
    # Color scale
    if fixed_scale:
        cmap_data = E
        cbar_label = '|E| induced'
        vmin, vmax = np.percentile(E, [5, 99])
    else:
        cmap_data = np.abs(E) * 100
        cbar_label = '|E| induced'
        vmin, vmax = 10**4, 10**8
    
    # Draw color map
    plt.pcolormesh(horizontal_axis, 
                   vertical_axis, 
                   cmap_data, 
                   cmap='inferno', 
                   shading='auto',
                   norm=LogNorm( vmin=vmin, vmax=50*vmax)
                      )
    print(f"h_axis{horizontal_axis.shape}", f"v_axis{vertical_axis.shape}", f"cmap{cmap_data.shape}")
    
#     plt.pcolormesh(horizontal_axis, vertical_axis, cmap_data[:-1, :-1],
#                 cmap='inferno', shading='auto',
#                 norm=LogNorm(vmin=vmin, vmax=100*vmax)
# )
#     plt.imshow(
#     cmap_data.T,
#     origin='lower',
#     aspect='auto',
#     extent=[ horizontal_axis.min(), horizontal_axis.max(),
#              vertical_axis.min(), vertical_axis.max()  ],
#     cmap='inferno',
#     norm=LogNorm(vmin=vmin, vmax=100*vmax)
# )

    plt.colorbar(label=cbar_label)
    
    # Streamlines
    plt.streamplot(horizontal_axis, vertical_axis, U, V,
                   density=density,
                   linewidth=(E - E.min()) / (E.max() - E.min()) + 0.05,
                   color='white',
                   arrowstyle=arrow)
    
    # plt.gca().set_aspect('equal', adjustable='box')
    # plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, 
                    # bbox_inches='tight'
                   )
        print(f"  Saved: {os.path.basename(save_path)}")
    
    plt.close()

In [9]:
def plot_Efield_stream(monitor_data, monitor_data0, monitor, peak, plane, 
                      normalize, fixed_scale,
                      density=2.5, arrow='fancy, head_length=0.7',
                      save_path=None):
    """Orchestrate field extraction and plotting for a 2D plane"""
    cfg = get_plane_config(plane)
    
    # Extract field components
    Ex = get_field_component(monitor_data, monitor_data0, 'Ex', cfg['slicer'], peak, normalize)
    Ey = get_field_component(monitor_data, monitor_data0, 'Ey', cfg['slicer'], peak, normalize)
    Ez = get_field_component(monitor_data, monitor_data0, 'Ez', cfg['slicer'], peak, normalize)
    
    # Get coordinates and prepare data
    coord1, coord2 = get_coords(monitor_data, plane)
    E, X, Y, Ex, Ey, Ez = prepare_Efield_data(Ex, Ey, Ez, coord1, coord2, plane)
    
    # Select vector components for streamlines
    field_map = {'Ex': Ex, 'Ey': Ey, 'Ez': Ez}
    U = field_map[cfg['U_field']]
    V = field_map[cfg['V_field']]
    
    # Plot
    plot_Efield(E, X, Y, U, V, cfg['xlabel'], cfg['ylabel'], plane, peak, 
               name, monitor, fixed_scale, density, arrow, save_path=save_path)


In [10]:
def process_single_monitor(monitor_name, peak, plane, save_path):

    # Extract only this monitor from both files
    monitor_data = extract_monitor_data(f'{save_dir}/{name_file}.hdf5', monitor_name)
    monitor_data0 = extract_monitor_data(f'{save_dir}/{name_file}_empty.hdf5', monitor_name)
    
    # Process and plot
    plot_Efield_stream(
        monitor_data, monitor_data0,
        monitor=monitor_name,
        peak=peak,
        plane=plane,
        normalize=False,
        fixed_scale=True, ##
        save_path=save_path
    )
    
    # Free memory
    del monitor_data, monitor_data0
    gc.collect()

# Execute

In [35]:
# --- DICTIONARY FOR IN-PLANE VALUES ---
ev_in_data = {
    'dipolEz_Th100D100G10':  [7.99, 16.19],
    'dipolEz_Th100D100G20':  [8.23, 16.45],
    'dipolEz_Th100D100G30':  [8.23, 16.52],
    'dipolEz_Th100D100G40':  [8.17, 16.52],
    'dipolEz_Th100D100G50':  [7.80, 15.81, 3.06],
    'dipolEz_Th100D100G100': [8.32, 17.01],
    'dipolEz_Th100D100G150': [8.27, 17.11],
    'dipolEz_Th100D100G200': [7.9, 15.89],
}

# --- DICTIONARY FOR OUT-PLANE VALUES ---
ev_out_data = {
    'dipolEz_Th100D100G10':  [8.05,16.17,  11.75],
    'dipolEz_Th100D100G20':  [8.29,16.4, ],
    'dipolEz_Th100D100G30':  [8.30, 16.39],
    'dipolEz_Th100D100G40':  [8.23, 16.45],
    'dipolEz_Th100D100G50':  [8.24,17.56],
    'dipolEz_Th100D100G100': [8.43, 13.59, 16.64, 19.29],
    'dipolEz_Th100D100G150': [8.37, 4.85],
    'dipolEz_Th100D100G200': [8.26],
}

In [36]:
# 5. Process in a row
save_dir = '/Users/Howfishy/Documents/0. tidy3d/2Mar Array/'

# List of the G-values you want to process
g_values = [
    # 10, 20, 
    # 30, 40, 
    50, 100, 
    # 150, 200
]

for g in g_values:
    name = f'dipolEz_Th100D100G{g}'
    
    if name not in ev_in_data:      # Check if this name exists in your dictionaries before processing
        print(f"Skipping {name}: No data found in dictionary.")
        continue

    print(f"--------------- Processing: {name} ------------------------")
    name_file = f"{name}_vertical_n1_nophase_in_out"  # Setup paths
    plot_dir = os.path.join(save_dir, f"field_plots_{name_file}")
    os.makedirs(plot_dir, exist_ok=True)

    # Process IN-PLANE
    # for EV in ev_in_data[name]:
    #     process_single_monitor(
    #         monitor_name='DFT_in_plane_slice0',
    #         peak=ev_to_hz(EV),
    #         plane='Exy',
    #         save_path=os.path.join(plot_dir, f'{EV:.2f}eV_{name}_IN_E.png')
    #     )

    # Process OUT-PLANE
    for EV in ev_out_data[name]:
        process_single_monitor(
            monitor_name='DFT_out_plane_XZ',
            peak=ev_to_hz(EV),
            plane='Exz',
            save_path=os.path.join(plot_dir, f'{EV:.2f}eV_OUT.png')
        )

--------------- Processing: dipolEz_Th100D100G50 ------------------------
(Ex.shape=(77, 278), expected=(278, 77))
[DEBUG] Exz-plane orientation check:
  horizontal_axis shape=(278, 77), vertical_axis shape=(278, 77)
  H-field array shape=(278, 77)
h_axis(278, 77) v_axis(278, 77) cmap(278, 77)
  Saved: 8.24eV_OUT.png
(Ex.shape=(77, 278), expected=(278, 77))
[DEBUG] Exz-plane orientation check:
  horizontal_axis shape=(278, 77), vertical_axis shape=(278, 77)
  H-field array shape=(278, 77)
h_axis(278, 77) v_axis(278, 77) cmap(278, 77)
  Saved: 17.56eV_OUT.png
--------------- Processing: dipolEz_Th100D100G100 ------------------------
(Ex.shape=(101, 278), expected=(278, 101))
[DEBUG] Exz-plane orientation check:
  horizontal_axis shape=(278, 101), vertical_axis shape=(278, 101)
  H-field array shape=(278, 101)
h_axis(278, 101) v_axis(278, 101) cmap(278, 101)
  Saved: 8.43eV_OUT.png
(Ex.shape=(101, 278), expected=(278, 101))
[DEBUG] Exz-plane orientation check:
  horizontal_axis shape=(27

In [29]:
# # --- SET THE ACTIVE NAME HERE ---
# name = 'dipolEz_Th100D100G20'
# # --------------------------------

# # 1. Setup Directories
# save_dir = '/Users/Howfishy/Documents/0. tidy3d/2Mar Array/'
# name_file = f"{name}_vertical_n1_nophase_in_out"
# plot_dir = os.path.join(save_dir, f"field_plots_{name_file}")
# os.makedirs(plot_dir, exist_ok=True)

# 2. Fetch data from dictionaries using the 'name' key
# .get() is safer as it won't crash if you mistype a name; it returns an empty list instead
# current_ev_in  = ev_in_data.get(name, [])

# 3. Process IN-PLANE monitors
# for EV in current_ev_in:
#     process_single_monitor(
#         monitor_name='DFT_in_plane_slice0',
#         peak=ev_to_hz(EV),
#         plane='Exy',
#         save_path=os.path.join(plot_dir, f'{EV:.2f}eV_{name}_IN_E.png')
#     )

# 4. Process OUT-PLANE monitors
# current_ev_out = ev_out_data.get(name, [])

# for EV in current_ev_out:
#     process_single_monitor(
#         monitor_name='DFT_out_plane_XZ',
#         peak=ev_to_hz(EV),
#         plane='Exz',
#         save_path=os.path.join(plot_dir, f'{EV:.2f}eV_OUT.png')
#     )

# print(f"Finished processing: {name}")

(Ex.shape=(61, 212), expected=(212, 61))
[DEBUG] Exy-plane orientation check:
  horizontal_axis shape=(212, 61), vertical_axis shape=(212, 61)
  H-field array shape=(212, 61)
h_axis(212, 61) v_axis(212, 61) cmap(212, 61)
  Saved: 8.23eV_dipolEz_Th100D100G20_IN_E.png
(Ex.shape=(61, 212), expected=(212, 61))
[DEBUG] Exy-plane orientation check:
  horizontal_axis shape=(212, 61), vertical_axis shape=(212, 61)
  H-field array shape=(212, 61)
h_axis(212, 61) v_axis(212, 61) cmap(212, 61)
  Saved: 16.45eV_dipolEz_Th100D100G20_IN_E.png
Finished processing: dipolEz_Th100D100G20


In [16]:
sim_data        = td.SimulationData.from_file(f'{save_dir}/{name}.hdf5')

sim_data.simulation.plot_3d()
print(sim_data.simulation.mediums)
for monitor in sim_data.simulation.monitors:
    print(monitor.name, monitor.type)

22:29:46 Malay Peninsula Standard Time WARNING: 'Simulation.mediums' will be    
                                       removed in Tidy3D 3.0. Use               
                                       'Simulation.scene.mediums' instead.      

[Air, Si_Palik_4-poles]
DFT_in_plane_slice0 FieldMonitor
DFT_in_plane_slice4.5 FieldMonitor


# Workings

In [27]:
# # save_dir = '/app/local_project/10Jan holes'
# save_dir = '/Users/Howfishy/Documents/0. tidy3d/2Mar Array/'
# os.makedirs(save_dir, exist_ok=True)

# # name      = 'dipolEz_Th100D100G10' 
# name      = 'dipolEz_Th100D100G20' 
# name_file = name + '_vertical_n1_nophase' + '_in_out'


# EV_Efield_in = [
#     # 7.99, 16.19,   # G10 
#     8.23, 16.45,   # G20
#     # 8.23, 16.52,   # G30
#     # 8.17, 16.52,   # G40
#     # 7.80, 15.81, 3.06,  # G50
#     # 8.32, 17.01,   # G100
#     # 8.27, 17.11,   # G150
#     # 7.900, 15.89   # G200
# ]  
    
# # In plane
# # 5.30, 8.28, 16.8              # 150   3.190894365, 14.85298868, 9.680526303
# # 5.27, 8.23, 16.82             # 100  3.043170514, 14.63900903, 9.291105512
# # 3.86, 8.32, 17.13            # 50    2.159042236, 12.30254829, 8.777088888
# # 3.78, 8.18, 13.34, 17.13  # 10  2.229249162, 9.612764384, 15.10925496, 11.30604998

# # Out
# # 9.25, 10.84, 18.42            # 150 6.632353266, 3.622842185, 12.44095544
# # 9.27, 11.55, 18.64            # 100 6.576221675, 8.15661918, 15.02129872
# # 9.33, 11.33, 16.16, 21.05  # 50  5.849506085, 7.045835761, 27.97736854, 10.75176837
# # 9.14, 12.03, 17.98             # 10  7.248261845, 7.622718546, 8.400214393




# os.makedirs('field_plots_'+ name_file, exist_ok=True)
# for EV in EV_Efield_in:
#     process_single_monitor(
#             monitor_name='DFT_in_plane_slice0',
#             peak=ev_to_hz(EV),
#             plane='Exy',
#             save_path=os.path.join('field_plots_' + name_file, f'{EV:.2f}eV_{name}_IN_E.png')
#         )
    
# for EV in EV_Efield_out:
#     process_single_monitor(
#             monitor_name='DFT_out_plane_XZ',
#             peak=ev_to_hz(EV),
#             plane='Exz',
#             save_path=os.path.join('field_plots_' + name_file, f'{EV:.2f}eV_OUT.png')
#         )

(Ex.shape=(56, 212), expected=(212, 56))
[DEBUG] Exy-plane orientation check:
  horizontal_axis shape=(212, 56), vertical_axis shape=(212, 56)
  H-field array shape=(212, 56)
h_axis(212, 56) v_axis(212, 56) cmap(212, 56)
  Saved: 7.99eV_dipolEz_Th100D100G10_IN_E.png
(Ex.shape=(56, 212), expected=(212, 56))
[DEBUG] Exy-plane orientation check:
  horizontal_axis shape=(212, 56), vertical_axis shape=(212, 56)
  H-field array shape=(212, 56)
h_axis(212, 56) v_axis(212, 56) cmap(212, 56)
  Saved: 16.19eV_dipolEz_Th100D100G10_IN_E.png


for suffix in [
               "_both_random",
               "_vertical_n1", 
               "_both_vertical", 
              ]:
name      = 'dipolEz_Th100D100G50'   
name_file = 'dipolEz_Th100D100G50' +suffix +'_withphase_in_plane'
name = 'dipolEz_D100L100' + '_OUT' #+ '_25'

In [ ]:
# Single (thickness vs diameter)
# EV_Efield = [
#     # 8.42, 16.38        # L100D100
#     # 8.27, 16.65             # Th100 D100 
#     # 10.35, 15.28       # L25D100 
#     # 10.19, 5.35, 20.57 # L100D200
#     # 5.051, 10.1376           #Th100 D200
#     # 10.24,             # L25D200 

#     # 10.26, 15.63 # Th25  D25
#     # 10.54, 15.28,  #Th25 D25_25
#     # 4.74, #12.43        # Th25 D50
#     # 9.1688, 16.65 # Th25 D100
#     # 8.85, 16.13  #Th25 D100_25
#     # 9.835, 4.85 # Th25 D200
#     # 8.399, 16.20  # D100 L100
# ]

# Dimers
EV_Efield = [
    # 8.40, 15.48, # G100
    # 8.39, 16.31, # G50  Ez  Both vert
    # 8.10, 17.15, # G10
    
    # 6.32,   # G100
    # 6.37,   # G50 Hx  Both vert
    # 6.45 ,  # G10  
    
    8.24, 16.73,  # G100
    # 8.26, 16.61,  # Gap50
    # 8.15, 16.64    # G10
]

In [12]:
EV_Efield_offset = [
    # 8.37, 15.99        # D100L100
    # 8.45, 16.01  # Th100 D100 
    # 10.17, 15.45       # D100 L25
    # 10.13, 5.398,           # D200L100
    # 10.23, 3.23, 18.33            # D200 L25

    3, 11.26 # Th25  D25

]

for EV in EV_Efield_offset:
    process_single_monitor(
            monitor_name='DFT_bottom_plane_slice14',
            peak=ev_to_hz(EV),
            plane='Exy',
            save_path=os.path.join('field_plots_' + name, f'{EV:.2f}eV_BOT_E.png')
        )

h_axis(42, 122) v_axis(42, 122) cmap(42, 122)
  Saved: 3.00eV_BOT.png
h_axis(42, 122) v_axis(42, 122) cmap(42, 122)
  Saved: 11.26eV_BOT.png


In [15]:
# Task
EV_Efield = 8.42
peak_freq = ev_to_hz(EV_Efield)

# Define all plotting tasks
tasks = [
    # ('DFT_in_plane_slice0',       'Exy', f'{EV_Efield:.2f}eV_IN_0.png'),
    # ('DFT_in_plane_slice4.5',     'Exy', f'{EV_Efield:.2f}eV_IN_4.5offset.png'),
    # ('DFT_bottom_plane_slice0.5', 'Exy', f'{EV_Efield:.2f}eV_BOT_0.5.png'),
    # ('DFT_bottom_plane_slice14',  'Exy', f'{EV_Efield:.2f}eV_BOT_14offset.png'),
    ('DFT_out_plane_XZ',          'Exz', f'{EV_Efield:.2f}eV_OUT_XZ.png'),
    ('DFT_out_plane_XZoffset',    'Exz', f'{EV_Efield:.2f}eV_OUT_XZoffset.png'),
]

# Process each monitor separately
for i, (monitor, plane, filename) in enumerate(tasks, 1):
    print(f"[{i}/{len(tasks)}]", end=" ")
    process_single_monitor(
        monitor_name=monitor,
        peak=peak_freq,
        plane=plane,
        save_path=os.path.join('field_plots_' + name, filename)
    )

[1/2]   Saved: 17.56eV_OUT_XZ.png
[2/2]   Saved: 17.56eV_OUT_XZoffset.png
